# Job Listing Prioritization for Career Changers: Time Series

PredictLeads × TripleTen externship · Jacob Colombo

## 0. Purpose

When do listings in the career-change target appear and close? Answers "when to search" and "how often to check", and creates timing features for the `ML_Rank` notebook.

**The series** (each notebook hands its final table to the next):

| Notebook | Question | Hands off |
|:--|:--|:--|
| EDA_SDA_1 | What's in the full file, how does tech compare with every field, and what's in the scope? (the scope cut, applied first) | Scope table and its descriptions, occupation summary, company activity |
| EDA_SDA_2 | Which scope listings do we keep, and how do they behave? Additional cleaning, non-text checks by role bucket, ghost listings | Scope table, non-text features |
| EDA_SDA_3 | What do the descriptions say? Text features, validated blockers, skills mapped to O*NET (no matching models) | Scope table + text features |
| EDA_SDA_4 | Which features hold up? Statistics, confounders, the adjusted Cox check | Modeling-ready table |
| TimeSeries | When do listings appear and close? | Timing features |
| ML_Fit | Which roles and listings fit a career changer? Keyword → TF-IDF → BGE → RRF, skills-based matching, a trained layer on real career moves | Fit scores, role discovery, the compass |
| ML_Rank | How should listings be ranked? The listing timing model, combined with fit into the priority queue | The priority queue algorithm |

**Standard:** the TripleTen course templates are the baseline (structure, methods, reporting). Every applicable course method is used, and taken further with more advanced techniques where the data calls for them.

### Clarifying questions
- How many new scope listings appear per day and week, and is there a weekly rhythm?
- Is that rhythm the employers' or the crawler's? (`posted`, read as Central European dates, vs. `first_seen`)
- Is there a trend across April–September? By role bucket, too: if the team adopts the proposed listing-momentum view (job growth tasks in `ANALYSIS.md`), each bucket's new-listings trend is shown next to expected growth, never blended into it (six months can't be backtested against employment). The full blend (4d) would draw its near-term signal from Indeed and JOLTS, which go back years, not from our six months.
- Can next-day or next-week volume be forecast better than a naive baseline?
- How fast do listings close in each month (survival by month)?
- Do listings open to career changers arrive on a different rhythm, for example entry-level hiring waves?

**Data rules:** aggregates only. Clear outputs before committing.

### Roadmap and methods

| Section | Course standard (TimeSeries template) | Taken further |
|:--|:--|:--|
| 1. Setup and load | Load the table, check it | Only the needed columns of EDA_SDA_2's scope table (memory); originals only (no reposts) |
| 2. The series | `resample()` to daily and weekly counts of new listings | Separate series from `first_seen` and from `posted` (CET), to separate crawler from employer rhythm; with and without job boards and likely ghosts; partial weeks marked, not dropped |
| 3. Analysis **before** features | Plot, rolling mean and std, decomposition (trend, seasonal, residual; weekly period), **ADF test** | STL; ACF / PACF; KPSS as a cross-check; differencing orders chosen from the tests |
| 4. Forecasting | Calendar features, **lags**, rolling mean and std **with `.shift()`**; train / validation / test in time order; **DummyRegressor**, LinearRegression, RandomForest with a tuning loop, ARIMA; **RMSE**; test used once | Three naive baselines (mean, last value, same weekday last week); SARIMA with weekly seasonality; gradient boosting; MAE; rolling-origin cross-validation; prediction intervals |
| 5. Survival and cadence | | Kaplan–Meier by month first seen (still-open listings censored); "check every N days" from the closing curve |
| 6. Role buckets | | Weekly new-listings trend per bucket (the listing-momentum view; shown, never blended into job growth) |
| 7. Conclusion and handoff | What was done, best model, limitations (six months of history) | Saved aggregates for `ML_Rank` and the app |

**Inputs and views.** Everything comes from EDA_SDA_2's scope table (`scope_table.parquet`), loaded column by column. Counts use **original listings only** (`is_repost` excluded). The two cuts that matter are shown side by side: **without likely ghosts** (the app hides them) and **without job boards** (they stop after May, so they bend any trend). Dates: `first_seen` follows the crawler (UTC days); `posted` is the employer's date, read as Central European dates. Only six months of data: **no seasonal claims**.

## 1. Setup and load

In [ ]:
# 1a Libraries and the shared helpers (src/eda_helpers.py: paths, survival curves, tables, charts;
# src/ts_helpers.py: daily and weekly counts, leakage-safe features, the time split, scores, check cadence)
import gc
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display
from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tools.sm_exceptions import ConvergenceWarning
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller, kpss

ROOT = next(d for d in [Path.cwd(), *Path.cwd().parents] if (d / "src" / "eda_helpers.py").exists())
sys.path.insert(0, str(ROOT))  # so `src` can be imported from notebooks/
from src.eda_helpers import (
    BLUE, ORANGE, AQUA, YELLOW, INK, INK_2, MUTED, CLEAN_DIR, PART2_TABLE,
    compact, finish, km_curve, km_median, pct, show, use_chart_style,
)
from src.ts_helpers import (
    CET, calendar_date, catch_rate, daily_counts, lag_features, longest_cadence, mae, rmse, time_split, weekly_counts,
)
use_chart_style()
RANDOM_STATE = 42
warnings.filterwarnings("ignore", category=ConvergenceWarning)  # SARIMA grid search: some orders won't converge
warnings.filterwarnings("ignore", category=UserWarning, module="statsmodels")

In [ ]:
# 1b Load only the columns this notebook needs from EDA_SDA_2's scope table (about 1M listings: memory matters)
# Originals only: a repost is another copy of an earlier listing, so it would count one opening twice.
NEEDED = ["first_seen", "posted", "days_open", "live", "role_bucket", "source_type", "is_repost", "likely_ghost"]
OPTIONAL = ["batch_first_seen"]  # a company added to the crawl in one batch is not a hiring surge (EDA_SDA_1 6.8)
if not PART2_TABLE.exists():
    raise FileNotFoundError("Run EDA_SDA_2 first: its section 4c saves the scope table.")
names = pq.read_schema(PART2_TABLE).names
missing = [c for c in NEEDED if c not in names]
if missing:
    raise KeyError(f"Scope table lacks {missing}; check the names in EDA_SDA_2's feature dictionary (4e).")
columns = NEEDED + [c for c in OPTIONAL if c in names]
raw = pq.read_table(PART2_TABLE, columns=columns).to_pandas(strings_to_categorical=True)

def as_flag(s):
    """True / false / unknown flags come back from Parquet as objects: unknown counts as false."""
    return s.astype("boolean").fillna(False).astype(bool)

for c in ["live", "is_repost", "likely_ghost", "batch_first_seen"]:
    if c in raw:
        raw[c] = as_flag(raw[c])
n_all = len(raw)
raw = raw[~raw.is_repost].drop(columns="is_repost").reset_index(drop=True)
print(f"{n_all:,} scope listings; {len(raw):,} originals (reposts excluded); columns loaded: {columns}")
print(f"{raw.memory_usage(deep=True).sum() / 1e9:.2f} GB in memory")

In [ ]:
# 1c Derived columns, then one slim frame `L`; the big frame is dropped
# The window starts April 1: March 30-31 are the crawl's first two days, when everything already open is "first seen"
DATA_START, DATA_END = pd.Timestamp("2026-04-01"), pd.Timestamp("2026-09-30")  # the data's first full day and end date
raw["seen_day"] = calendar_date(raw.first_seen)          # the crawler's day (UTC)
raw["posted_day"] = calendar_date(raw.posted, CET)       # the employer's day (midnight Central European time)
raw["job_board"] = raw.source_type.astype(str).str.contains("board", case=False)  # job boards stop after May (EDA_SDA_2 3.2c)
d = raw.days_open
d = d.dt.total_seconds() / 86400 if pd.api.types.is_timedelta64_dtype(d) else d.astype(float)
raw["days"] = np.floor(d).clip(lower=0).astype("float32")  # whole days open; NaN stays NaN
raw["finished"] = ~raw.live                                # live = still open on Sept 30: censored, not closed
raw["role_bucket"] = raw.role_bucket.astype("object").fillna("unmapped").astype("category")
in_window = raw.seen_day.between(DATA_START, DATA_END)
keep = ["seen_day", "posted_day", "role_bucket", "job_board", "likely_ghost", "days", "finished"]
keep += [c for c in ["batch_first_seen"] if c in raw]
L = raw.loc[in_window, keep].reset_index(drop=True)
L["week"] = L.seen_day - pd.to_timedelta(L.seen_day.dt.dayofweek, unit="D")  # the Monday of the week first seen
L["month"] = L.seen_day.dt.strftime("%Y-%m")
source_mix = raw.loc[in_window, "source_type"].value_counts().to_frame("listings")  # to check the job-board pattern
del raw
gc.collect()
show("1c The analysis frame", pd.DataFrame({"value": {
    "original listings first seen April 1 - September 30": compact(len(L)),
    "not used: reposts, and first seen March 30-31 (crawl start)": compact(n_all - len(L)),
    "with an employer posting date": pct(L.posted_day.notna().mean()),
    "from job boards": pct(L.job_board.mean()),
    "likely ghosts": pct(L.likely_ghost.mean()),
    "still open on September 30 (censored)": pct((~L.finished).mean()),
    "days open missing": pct(L.days.isna().mean()),
}}))
print("Source types (job boards are the labels containing 'board'):")
display(source_mix)

## 2. The series: new listings per day and per week

Two clocks. **`first_seen`** is when the crawler found the listing, so it follows the crawler's schedule. **`posted`** is the employer's own date (about 40% of listings have one), so it follows employers. If a weekly pattern shows in both, it is real hiring behavior; if only in `first_seen`, it is the crawler. The comparison is the day-of-week rule in `STANDARDS.md`.

In [ ]:
# 2a Daily counts of new original listings, several views. Days with no listings are zeros, not gaps.
fs_all = L.seen_day
series = pd.DataFrame({
    "first_seen_all": daily_counts(fs_all, DATA_START, DATA_END),
    "first_seen_no_ghost": daily_counts(fs_all[~L.likely_ghost], DATA_START, DATA_END),  # what the app shows
    "first_seen_no_board": daily_counts(fs_all[~L.job_board], DATA_START, DATA_END),     # steady collection method
    "posted_all": daily_counts(L.posted_day, DATA_START, DATA_END),                      # employer clock (CET dates)
})
if "batch_first_seen" in L:
    series["first_seen_no_batch"] = daily_counts(fs_all[~L.batch_first_seen], DATA_START, DATA_END)
weekly = pd.concat([weekly_counts(series[c])[c] for c in series], axis=1)
weekly["full_week"] = weekly_counts(series["first_seen_all"])["full_week"]
show("2a Days and weeks in the series", pd.DataFrame({"value": {
    "days": len(series), "weeks (Monday-Sunday)": len(weekly), "full weeks": int(weekly.full_week.sum()),
    "days with no new listings (first_seen)": int((series.first_seen_all == 0).sum()),
    "first / last week partial": "yes: April 1 is a Wednesday, September 30 a Wednesday",
}}))
display(series.describe().round(0).T)

In [ ]:
# 2b Daily and weekly new listings from first_seen. Partial weeks are drawn in grey, not dropped.
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
daily = series["first_seen_all"]
axes[0].plot(daily.index, daily, color=MUTED, lw=1, label="daily")
axes[0].plot(daily.index, daily.rolling(7).mean(), color=BLUE, label="7-day mean")
axes[0].set(title="New listings per day (first seen)", ylabel="listings")
axes[0].legend()
wk = weekly["first_seen_all"]
axes[1].bar(wk.index, wk, width=5.5, color=[BLUE if f else MUTED for f in weekly.full_week])
axes[1].set(title="New listings per week (grey: partial week)", ylabel="listings")
finish()

*Comment (after the run): level, any step change, the weekly rhythm in the daily panel, and what the grey partial weeks hide.*

In [ ]:
# 2c The same by the employer's posting date (CET dates), where the listing has one
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.bar(weekly.index, weekly["posted_all"], width=5.5, color=[ORANGE if f else MUTED for f in weekly.full_week])
ax.set(title="Listings per week by employer posting date (grey: partial week)", ylabel="listings with a posting date")
finish()

*Comment (after the run): does the employer's weekly count follow the crawler's? Posting dates near September 30 undercount, because a listing posted but not yet crawled is not in the table.*

In [ ]:
# 2d Crawler rhythm vs. employer rhythm: share of each week's listings by weekday, full weeks only
week_of_day = pd.Series(series.index - pd.to_timedelta(series.index.dayofweek, unit="D"), index=series.index)
in_full = week_of_day.map(weekly.full_week).fillna(False).astype(bool)
by_day = series.loc[in_full, ["first_seen_all", "posted_all"]].groupby(series.index[in_full].dayofweek).sum()
share = by_day / by_day.sum()
labels = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
x = np.arange(7)
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.bar(x - 0.2, share.first_seen_all, 0.4, color=BLUE, label="first seen (crawler, UTC days)")
ax.bar(x + 0.2, share.posted_all, 0.4, color=ORANGE, label="posted (employer, CET dates)")
ax.axhline(1 / 7, color=INK_2, ls="--", lw=1, label="even split (1/7)")
ax.set(title="Share of weekly listings by weekday", xticks=x, xticklabels=labels, ylabel="share of the week")
ax.legend()
finish()
show("2d Share of the week by weekday", share.rename(columns={"first_seen_all": "first seen share", "posted_all": "posted share"})
     .set_axis(labels))

*Comment (after the run): which weekdays are heavy on each clock. A weekend dip only in `posted` is employer behavior; a pattern only in `first_seen` is the crawler's schedule.*

In [ ]:
# 2e What bends the series? Job boards stop after May, and ghosts first seen early are the ones filtered out.
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8), sharey=True)
w_board = L[L.job_board].groupby("week").size().reindex(weekly.index, fill_value=0)
w_other = L[~L.job_board].groupby("week").size().reindex(weekly.index, fill_value=0)
axes[0].stackplot(weekly.index, w_other, w_board, colors=[BLUE, YELLOW], labels=["company sites and platforms", "job boards"])
axes[0].set(title="New listings per week by source type", ylabel="listings")
axes[0].legend(loc="upper right")
for c, color, lab in [("first_seen_all", BLUE, "all originals"), ("first_seen_no_ghost", AQUA, "without likely ghosts"),
                      ("first_seen_no_board", ORANGE, "without job boards")]:
    axes[1].plot(weekly.index[weekly.full_week], weekly.loc[weekly.full_week, c], color=color, label=lab)
axes[1].set(title="Weekly new listings, three views (full weeks)")
axes[1].legend()
finish()
by_month = L.groupby("month").agg(listings=("seen_day", "size"), **{"job board share": ("job_board", "mean"),
                                  "likely ghost share": ("likely_ghost", "mean"), "has posting date share": ("posted_day", lambda s: s.notna().mean())})
show("2e Mix by month first seen", by_month)

*Comment (after the run): the job-board share by month shows where the collection changed; any step in the weekly count at that point is the collection, not the market. Likely ghosts are, by definition, over 90 days old and open on September 30, so they were all first seen before July: dropping them lowers the early months and tilts the trend upward. Trend claims therefore use the all-originals and no-job-board views; the no-ghost view is what the app shows.*

**What the data shows**
- (fill in after the run: level and range of new listings per day and per week)
- (fill in after the run: weekday shares on each clock, and whether the weekly rhythm is the crawler's or the employers')
- (fill in after the run: when the job-board share falls and what it does to the weekly count)
- (fill in after the run: days with zero listings, if any, and what they look like)

## 3. Analysis before features

**Series used from here:** daily new original listings by `first_seen`, all sources (`SERIES`). That is what a career changer meets in the app. Set it to `"first_seen_no_board"` to repeat sections 3 and 4 on the steadier collection method (decide after 2e).

In [ ]:
# 3a The series, and its rolling mean and standard deviation
SERIES = "first_seen_all"
y = series[SERIES].astype(float)
fig, axes = plt.subplots(2, 1, figsize=(11, 5.5), sharex=True)
axes[0].plot(y.index, y, color=MUTED, lw=1, label="daily")
axes[0].plot(y.index, y.rolling(7).mean(), color=BLUE, label="rolling mean, 7 days")
axes[0].plot(y.index, y.rolling(28).mean(), color=ORANGE, label="rolling mean, 28 days")
axes[0].set(title=f"New listings per day ({SERIES})", ylabel="listings")
axes[0].legend(ncol=3)
axes[1].plot(y.index, y.rolling(7).std(), color=BLUE, label="rolling std, 7 days")
axes[1].plot(y.index, y.rolling(28).std(), color=ORANGE, label="rolling std, 28 days")
axes[1].set(title="Rolling standard deviation", ylabel="listings")
axes[1].legend()
finish()

*Comment (after the run): a drifting mean or a std that grows with the level means the series is not stationary and may need a log or a difference.*

In [ ]:
# 3b STL decomposition with a weekly period (robust: one odd day does not bend the trend)
stl = STL(y, period=7, robust=True).fit()
fig, axes = plt.subplots(4, 1, figsize=(11, 8), sharex=True)
for ax, (name, comp, color) in zip(axes, [("observed", y, INK_2), ("trend", stl.trend, BLUE),
                                          ("weekly seasonal", stl.seasonal, ORANGE), ("remainder", stl.resid, MUTED)]):
    ax.plot(comp.index, comp, color=color)
    ax.set_title(name)
finish()
# Strength of each part (Hyndman): 0 = none, 1 = all of the variation
strength = {"trend": max(0, 1 - stl.resid.var() / (stl.trend + stl.resid).var()),
            "weekly seasonal": max(0, 1 - stl.resid.var() / (stl.seasonal + stl.resid).var())}
show("3b Strength of the STL components", pd.Series(strength, name="strength (0-1)"))

*Comment (after the run): trend line vs. the 28-day mean; the size of the weekly swing in listings; whether the remainder has spikes (a company batch, a crawler outage).*

In [ ]:
# 3c Autocorrelation: lags 1-35 days. Spikes at 7, 14, 21 mean a weekly rhythm.
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
plot_acf(y, lags=35, ax=axes[0], color=BLUE, title="ACF of new listings per day")
plot_pacf(y, lags=35, ax=axes[1], color=BLUE, method="ywm", title="PACF of new listings per day")
finish()

*Comment (after the run): which lags stand out; this guides the lag features in section 4 (1, 2, 3, 7, 14).*

In [ ]:
# 3d Stationarity: ADF (null: not stationary) and KPSS (null: stationary), on the level and on differenced versions.
# Both are read together: ADF p < 0.05 and KPSS p > 0.05 means stationary. KPSS p-values are capped at 0.01 and 0.10.
def stationarity(x):
    x = x.dropna()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")  # statsmodels announces a coming change in what adfuller returns
        adf_p = adfuller(x, autolag="AIC")[1]
        kpss_p = kpss(x, regression="c", nlags="auto")[1]
    if adf_p < 0.05 and kpss_p > 0.05:
        verdict = "stationary"
    elif adf_p >= 0.05 and kpss_p <= 0.05:
        verdict = "not stationary"
    elif adf_p < 0.05:
        verdict = "ADF yes, KPSS no (trend-stationary?)"
    else:
        verdict = "inconclusive"
    return {"ADF p": adf_p, "KPSS p": kpss_p, "verdict": verdict}

VARIANTS = {"level": y, "difference (1 day)": y.diff(), "seasonal difference (7 days)": y.diff(7),
            "both": y.diff().diff(7)}
tests = pd.DataFrame({name: stationarity(v) for name, v in VARIANTS.items()}).T
tests[["ADF p", "KPSS p"]] = tests[["ADF p", "KPSS p"]].astype(float).round(3)
print("3d Stationarity tests")
display(tests)
# Orders for SARIMA, from the evidence: seasonal difference if the weekly part is strong (Hyndman: strength > 0.64),
# then an ordinary difference only if what is left still fails both tests.
D_SEAS = int(strength["weekly seasonal"] > 0.64)
d_ORDER = 0 if tests.loc["seasonal difference (7 days)" if D_SEAS else "level", "verdict"] == "stationary" else 1
print(f"Chosen differencing: d = {d_ORDER}, seasonal D = {D_SEAS} (period 7)")

*Comment (after the run): what ADF and KPSS agree on for the level, and which differencing makes the series stationary. Six months is short, so a weak trend can pass for a unit root; the two tests together are a guide, not a proof.*

**What the data shows**
- (fill in after the run: trend direction and size from STL, and its strength)
- (fill in after the run: weekly seasonal strength and the heaviest and lightest weekday)
- (fill in after the run: ACF / PACF lags that matter)
- (fill in after the run: ADF and KPSS verdicts and the chosen differencing)

## 4. Forecasting new listings per day

**Question:** can tomorrow's count be forecast better than a naive guess? Useful for the app's "expect about N new listings" line, and a check that the timing signal is real before `ML_Rank` uses it.

| Decision | Choice | Why |
|:--|:--|:--|
| Target | New original listings per day (`SERIES`) | What the app sees |
| Features | Lags 1, 2, 3, 7, 14; rolling mean and std over 7 and 14 days; weekday; US federal holiday | Each lag and rolling statistic is shifted one day, so day t never sees itself; weekday and holiday are known ahead |
| Split | Train April - July, validate August, test September; no shuffling | Time order; the test month is used once, at the end (4h) |
| Baselines | Training mean (`DummyRegressor`), last value (yesterday), same weekday last week | Beating "yesterday" and "last week" is the real bar |
| Models | Linear regression, random forest (tuned on validation), gradient boosting, SARIMA | Course standard plus weekly seasonality |
| Scores | RMSE (listings; large misses count more) and MAE | Reported on validation, then rolling-origin cross-validation (mean ± std) |
| Forecast horizon | One day ahead, from actual history | Every model is scored the same way; SARIMA uses its fixed parameters on the actual past |

In [ ]:
# 4a Features (lags and rolling statistics from the past only, via .shift) and the time-ordered split
X, y_f = lag_features(y)
VAL_START, TEST_START = "2026-08-01", "2026-09-01"
train, val, test = time_split(X.index, VAL_START, TEST_START)
show("4a Features and split", pd.DataFrame({"value": {
    "features": X.shape[1], "train (April 15 - July)": f"{train.sum()} days", "validation (August)": f"{val.sum()} days",
    "test (September, untouched until 4h)": f"{test.sum()} days"}}))
print(", ".join(X.columns))

In [ ]:
# 4b Every model as a function: fit on the days in `fit`, predict the days in `pred` (boolean masks on X's index)
def ml(model):
    def run(fit, pred):
        m = clone(model).fit(X[fit], y_f[fit])
        return pd.Series(m.predict(X[pred]), index=X.index[pred])
    return run

FORECASTERS = {  # baselines first
    "Baseline: training mean": ml(DummyRegressor(strategy="mean")),
    "Baseline: last value (yesterday)": lambda fit, pred: X.loc[pred, "lag_1"],
    "Baseline: same weekday last week": lambda fit, pred: X.loc[pred, "lag_7"],
    "Linear regression": ml(Pipeline([("scale", StandardScaler()), ("lr", LinearRegression())])),
}
BASELINES = [k for k in FORECASTERS if k.startswith("Baseline")]

def score_table(preds, truth):
    """RMSE and MAE per model, best RMSE first."""
    t = pd.DataFrame({name: {"RMSE": rmse(truth.loc[p.index], p), "MAE": mae(truth.loc[p.index], p)} for name, p in preds.items()}).T
    return t.sort_values("RMSE")

In [ ]:
# 4c Random forest with a tuning loop on the validation month (the course's loop; the grid is small on purpose)
tuning = []
for n_trees in (200, 500):
    for depth in (3, 5, 8, None):
        for leaf in (1, 3, 5):
            rf = RandomForestRegressor(n_estimators=n_trees, max_depth=depth, min_samples_leaf=leaf,
                                       random_state=RANDOM_STATE, n_jobs=-1)
            p = ml(rf)(train, val)
            tuning.append({"trees": n_trees, "max depth": depth, "min leaf": leaf, "validation RMSE": rmse(y_f[val], p)})
tuning = pd.DataFrame(tuning).sort_values("validation RMSE").reset_index(drop=True)
best_rf = tuning.iloc[0]
print("4c Best random forest settings (of", len(tuning), "tried)")
display(tuning.head(5).round(2))
BEST_RF = RandomForestRegressor(n_estimators=int(best_rf["trees"]),
                                max_depth=None if pd.isna(best_rf["max depth"]) else int(best_rf["max depth"]),
                                min_samples_leaf=int(best_rf["min leaf"]), random_state=RANDOM_STATE, n_jobs=-1)
FORECASTERS["Random forest"] = ml(BEST_RF)
FORECASTERS["Gradient boosting"] = ml(HistGradientBoostingRegressor(
    max_depth=3, learning_rate=0.05, max_iter=300, random_state=RANDOM_STATE))

In [ ]:
# 4d SARIMA with weekly seasonality. d and D come from section 3; p, q, P, Q by AIC on the training months only
# (AIC is comparable here because d and D are fixed).
def fit_sarima(endog, order, seasonal):
    return SARIMAX(endog, order=order, seasonal_order=seasonal, enforce_stationarity=False,
                   enforce_invertibility=False).fit(disp=False, maxiter=200)

y_train = y[y.index < pd.Timestamp(VAL_START)]
grid = []
for p in range(3):
    for q in range(3):
        for P in range(2):
            for Q in range(2):
                try:
                    res = fit_sarima(y_train, (p, d_ORDER, q), (P, D_SEAS, Q, 7))
                    grid.append({"p": p, "q": q, "P": P, "Q": Q, "AIC": res.aic})
                except Exception:  # a few orders fail to fit on 100-odd days: skip them
                    continue
sarima_grid = pd.DataFrame(grid).sort_values("AIC").reset_index(drop=True)
b = sarima_grid.iloc[0]
SARIMA_ORDER = (int(b.p), d_ORDER, int(b.q))
SARIMA_SEASONAL = (int(b.P), D_SEAS, int(b.Q), 7)
print(f"4d Best SARIMA order {SARIMA_ORDER} x {SARIMA_SEASONAL} by AIC (of {len(sarima_grid)} fitted)")

def sarima_forecast(fit, pred):
    """Parameters fitted on the days up to the end of `fit`; one-step-ahead predictions over `pred` from the actual past."""
    fit_end = X.index[fit].max()
    res = fit_sarima(y[y.index <= fit_end], SARIMA_ORDER, SARIMA_SEASONAL)
    full = res.apply(y[y.index <= X.index[pred].max()])  # same parameters, extended with the actual days
    pm = full.get_prediction(start=X.index[pred].min()).predicted_mean
    return pm.reindex(X.index[pred]).clip(lower=0)

FORECASTERS["SARIMA"] = sarima_forecast

In [ ]:
# 4e Validation (August): every model, fitted on April-July
val_preds = {name: f(train, val) for name, f in FORECASTERS.items()}
val_scores = score_table(val_preds, y_f)
print("4e Validation scores, August (listings per day; lower is better)")
display(val_scores.round(1))
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(y_f[val].index, y_f[val], color=INK, label="actual")
best_val = val_scores.drop(index=BASELINES).index[0]
for name, color in [("Baseline: same weekday last week", MUTED), (best_val, BLUE)]:
    ax.plot(val_preds[name].index, val_preds[name], color=color, label=name)
ax.set(title="August: actual vs. forecast, one day ahead", ylabel="listings per day")
ax.legend()
finish()

*Comment (after the run): which model wins on validation and by how much over "same weekday last week". If nothing clearly beats the baselines, say so: the forecast adds little over a weekday rule.*

In [ ]:
# 4f Rolling-origin cross-validation on April-August (the test month stays out): 5 folds of 14 days, each fitted
# only on what came before. Mean ± std of the RMSE per model, so a lucky month cannot decide.
dev = np.where(train | val)[0]
cv_rows = {name: [] for name in FORECASTERS}
for fit_pos, pred_pos in TimeSeriesSplit(n_splits=5, test_size=14).split(dev):
    fit_mask = np.zeros(len(X), bool); fit_mask[dev[fit_pos]] = True
    pred_mask = np.zeros(len(X), bool); pred_mask[dev[pred_pos]] = True
    for name, f in FORECASTERS.items():
        cv_rows[name].append(rmse(y_f[pred_mask], f(fit_mask, pred_mask)))
cv = pd.DataFrame({name: {"CV RMSE mean": np.mean(v), "CV RMSE std": np.std(v, ddof=1)} for name, v in cv_rows.items()}).T
print("4f Rolling-origin cross-validation (5 folds x 14 days, April-August)")
display(cv.sort_values("CV RMSE mean").round(1))

*Comment (after the run): does the validation ranking hold across folds, and is the gap to the baselines larger than the fold-to-fold spread?*

In [ ]:
# 4g Prediction intervals for the model chosen on validation: the 10th and 90th percentile of its validation errors
# (empirical, no distribution assumed). Their coverage is checked on the test month in 4h.
chosen = best_val  # chosen BEFORE the test month is touched: the best non-baseline RMSE on validation
resid_val = y_f[val] - val_preds[chosen]
lo_q, hi_q = resid_val.quantile(0.10), resid_val.quantile(0.90)
print(f"Chosen on validation: {chosen}. 80% interval: forecast {lo_q:+.0f} to {hi_q:+.0f} listings")

In [ ]:
# 4h THE TEST MONTH (September), RUN ONCE. Nothing above was tuned on it. The chosen model and the baselines are
# refitted on April-August and scored on September, then the cell is not run again.
fit_all = train | val
test_preds = {name: FORECASTERS[name](fit_all, test) for name in BASELINES + [chosen]}
test_scores = score_table(test_preds, y_f)
print(f"4h Test scores, September (listings per day). Chosen model: {chosen}")
display(test_scores.round(1))
p = test_preds[chosen]
covered = ((y_f[test] >= p + lo_q) & (y_f[test] <= p + hi_q)).mean()
print(f"The 80% interval covered {pct(covered)} of September's days")
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(y_f[test].index, y_f[test], color=INK, label="actual")
ax.plot(p.index, p, color=BLUE, label=chosen)
ax.fill_between(p.index, (p + lo_q).clip(lower=0), p + hi_q, color=BLUE, alpha=0.15, label="80% interval")
ax.plot(test_preds["Baseline: same weekday last week"].index, test_preds["Baseline: same weekday last week"], color=MUTED,
        label="same weekday last week")
ax.set(title="September (test): actual vs. forecast", ylabel="listings per day")
ax.legend()
finish()

*Comment (after the run): the chosen model's test RMSE and MAE against the baselines, in listings per day and as a share of the average daily count. Compare with its validation score: a large gap means validation was lucky. September also ends on the data's last day, so the final days can be thin if the crawl lags.*

**What the data shows**
- (fill in after the run: validation winner, RMSE and MAE vs. the three baselines)
- (fill in after the run: cross-validation mean ± std, and whether the ranking holds)
- (fill in after the run: test RMSE and MAE of the chosen model vs. the baselines, interval coverage)
- (fill in after the run: the best SARIMA order and whether weekly seasonality was worth modeling)
- **Limits:** six months, one crawler, job boards stopping after May: the model describes this period, not a season.

## 5. How long listings stay open, and how often to check

**Survival by month first seen.** Kaplan–Meier on days open, with listings still open on September 30 **censored** (we know they lasted at least that long, not when they close). Later months have less follow-up, so each curve stops at the days every listing in that month could have been followed.

**Cadence.** Someone who checks every N days meets a new listing U days after it appears, with U between 0 and N (equally likely). They **catch** it if it is still open then, and with `k` days left to apply if it is still open at U + k. The catch rate is the average of the closing curve over that window.

In [ ]:
# 5a Kaplan–Meier by month first seen
MONTHS = sorted(L.month.unique())
colors = [BLUE, ORANGE, AQUA, YELLOW, INK_2, MUTED]
fig, ax = plt.subplots(figsize=(8.5, 4.5))
rows = {}
for m, color in zip(MONTHS, colors):
    g = L[L.month == m]
    curve = km_curve(g.days, g.finished)
    month_end = pd.Period(m).end_time.normalize()
    horizon = int(min(90, max(14, (DATA_END - month_end).days)))  # follow-up every listing of the month has
    shown = curve[curve.index <= horizon]
    ax.step(shown.index, shown.values, where="post", color=color, label=f"{m} (n={compact(len(g))})")
    at = lambda t: float(curve[curve.index <= t].iloc[-1]) if t <= horizon and (curve.index <= t).any() else np.nan
    rows[m] = {"listings": len(g), "still open after 7 days": at(7), "still open after 14 days": at(14),
               "still open after 30 days": at(30), "median days open": km_median(g.days, g.finished),
               "still open Sept 30 (censored)": 1 - g.finished.mean()}
ax.set(title="Share of listings still open, by month first seen", xlabel="days since first seen", ylim=(0, 1.02))
ax.legend()
finish()
show("5a Closing speed by month first seen (blank: not enough follow-up)", pd.DataFrame(rows).T)

*Comment (after the run): do the monthly curves overlap or does closing speed change by month? A later month closing faster can be real, or the crawler losing sites; check against the job-board stop (2e) before reading it as the market.*

In [ ]:
# 5b "Check every N days": catch rates from the closing curve. Two views: everything, and what the app shows
# (no likely ghosts). Ghosts are mostly censored long-open listings, so the app's curve closes faster.
curves = {"all originals": km_curve(L.days, L.finished),
          "what the app shows (no likely ghosts)": km_curve(L.days[~L.likely_ghost], L.finished[~L.likely_ghost])}
EVERY, LEFT = (1, 2, 3, 5, 7, 10, 14, 21, 30), (0, 3, 7)
cadence = {}
for view, curve in curves.items():
    t = pd.DataFrame({f"{k} extra days to apply": [catch_rate(curve, n, k) for n in EVERY] for k in LEFT},
                     index=pd.Index(EVERY, name="check every N days"))
    cadence[view] = t
    print(f"5b Share of new listings you see still open, checking every N days: {view}")
    display(t.apply(lambda s: s.map(pct)))
fig, ax = plt.subplots(figsize=(8, 4.2))
for k, color in zip(LEFT, [BLUE, ORANGE, AQUA]):
    t = cadence["what the app shows (no likely ghosts)"]
    ax.plot(t.index, t[f"{k} extra days to apply"], color=color, marker="o", label=f"with {k} extra days to apply")
ax.axhline(0.8, color=INK_2, ls="--", lw=1)
ax.set(title="Share of listings caught, by how often you check", xlabel="check every N days", ylim=(0, 1.02))
ax.legend()
finish()

*Comment (after the run): where each line crosses the dashed 80% line is the longest gap between checks that still catches four in five listings.*

In [ ]:
# 5c The longest gap between checks that still catches 90 / 80 / 70% of listings, overall and by role bucket
curve_app = curves["what the app shows (no likely ghosts)"]
longest = pd.DataFrame({f"{int(t * 100)}% caught": {f"{k} extra days to apply": longest_cadence(curve_app, t, k) for k in LEFT}
                        for t in (0.9, 0.8, 0.7)})
print("5c Longest gap between checks (days) that still catches the share, with k extra days to apply")
display(longest)
app = L[~L.likely_ghost]
by_bucket = {}
for bucket, g in app.groupby("role_bucket", observed=True):
    if len(g) < 1000:  # too few for a stable curve
        continue
    c = km_curve(g.days, g.finished)
    by_bucket[bucket] = {"listings": len(g), "median days open": km_median(g.days, g.finished),
                         "caught checking every 3 days (3 days to apply)": catch_rate(c, 3, 3),
                         "caught checking every 7 days (3 days to apply)": catch_rate(c, 7, 3),
                         "longest gap for 80% (3 days to apply)": longest_cadence(c, 0.8, 3) or np.nan}
by_bucket = pd.DataFrame(by_bucket).T.astype(float).sort_values("listings", ascending=False)
show("5c Check cadence by role bucket (what the app shows)", by_bucket)

*Comment (after the run): does the answer differ by bucket enough to give a different "check every N days" per role, or is one number fine?*

**What the data shows**
- (fill in after the run: share still open after 7, 14 and 30 days, and median days open, by month)
- (fill in after the run: whether closing speed moved between months, and what else could explain it)
- (fill in after the run: the recommended gap between checks, e.g. "check every N days to catch 80% with 3 days to apply")
- (fill in after the run: buckets whose cadence differs from the overall one)
- **Limits:** `days_open` ends at the crawler's last sighting, which can come before the real closing (and a crawl dropout ends it early), so true time left is a little longer than shown.

## 6. New listings per week by role bucket (listing momentum)

Each bucket's weekly count of new listings, for the "listing momentum" view next to expected job growth (BLS). **Shown beside growth, never blended into it:** six months cannot be backtested against employment, and the full growth blend would take its near-term signal from Indeed and JOLTS, which go back years. Two views per bucket, all sources and without job boards, because the job-board stop after May moves some buckets more than others.

In [ ]:
# 6a Weekly new listings per bucket, same weeks as section 2 (full weeks drawn solid, partial weeks hollow)
def weekly_by_bucket(frame):
    w = frame.groupby(["week", "role_bucket"], observed=True).size().unstack(fill_value=0).reindex(weekly.index, fill_value=0)
    w.index.name = "week"
    return w

wk_all, wk_nb = weekly_by_bucket(L), weekly_by_bucket(L[~L.job_board])
order = wk_all.sum().sort_values(ascending=False).index
order = [b for b in order if wk_all[b].sum() > 0]
ncols = 3
nrows = -(-len(order) // ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(13, 2.6 * nrows), sharex=True, squeeze=False)
full = weekly.full_week
for ax, bucket in zip(axes.ravel(), order):
    ax.plot(weekly.index[full], wk_all.loc[full, bucket], color=BLUE, label="all sources")
    ax.plot(weekly.index[full], wk_nb.loc[full, bucket], color=ORANGE, ls="--", label="without job boards")
    ax.scatter(weekly.index[~full], wk_all.loc[~full, bucket], facecolors="none", edgecolors=MUTED, zorder=3)
    ax.set_title(str(bucket), fontsize=9.5)
for ax in axes.ravel()[len(order):]:
    ax.axis("off")
axes[0, 0].legend(fontsize=8)
fig.suptitle("New listings per week by role bucket (hollow: partial week)", x=0.01, ha="left", fontweight="bold")
finish()

*Comment (after the run): which buckets rise, fall or stay flat, and whether the dashed (no job board) line tells the same story as the solid one.*

In [ ]:
# 6b Momentum per bucket: last 4 full weeks vs. first 4 full weeks, and a straight-line trend per week.
# The sign check is the guard: where the two views disagree, the job-board stop is driving the "trend".
def momentum(w):
    f = w[full].astype(float)
    slope = np.polyfit(np.arange(len(f)), f.values, 1)[0] / f.mean() if f.mean() > 0 else np.nan
    first4, last4 = f.iloc[:4].mean(), f.iloc[-4:].mean()
    return {"avg per week": f.mean(), "change, last vs first 4 weeks": last4 / first4 - 1 if first4 > 0 else np.nan,
            "trend per week": slope}

mom = pd.concat({"all sources": pd.DataFrame({b: momentum(wk_all[b]) for b in order}).T,
                 "without job boards": pd.DataFrame({b: momentum(wk_nb[b]) for b in order}).T}, axis=1)
mom["views agree"] = np.sign(mom[("all sources", "trend per week")]) == np.sign(mom[("without job boards", "trend per week")])
fmt = mom.copy()
for view in ("all sources", "without job boards"):
    fmt[(view, "avg per week")] = mom[(view, "avg per week")].map(lambda v: f"{v:,.0f}")
    fmt[(view, "change, last vs first 4 weeks")] = mom[(view, "change, last vs first 4 weeks")].map(lambda v: f"{v:+.0%}")
    fmt[(view, "trend per week")] = mom[(view, "trend per week")].map(lambda v: f"{v:+.1%}")
print("6b Listing momentum by role bucket (full weeks only)")
display(fmt)

*Comment (after the run): buckets whose views disagree are not given a momentum label.*

**What the data shows**
- (fill in after the run: the buckets with rising and falling weekly listings, in both views)
- (fill in after the run: buckets where the job-board stop changes the story)
- (fill in after the run: weekly volume per bucket, the largest and smallest)
- **Limits:** new listings measure openings posted, not net hiring; a bucket with falling listings can still grow in jobs. These are shown next to the BLS growth, never added to it.

## 7. Conclusion and handoff

In [ ]:
# 7a Save the aggregates (counts per day, week and bucket; no listing rows) for ML_Rank and the app
daily_out = series.reset_index()
weekly_bucket = (wk_all.stack().rename("new_listings_all").to_frame()
                 .join(wk_nb.stack().rename("new_listings_no_job_board")).reset_index())
weekly_bucket["role_bucket"] = weekly_bucket["role_bucket"].astype(str)
weekly_bucket["full_week"] = weekly_bucket["week"].map(weekly.full_week)
cadence_out = pd.concat({k: v.reset_index() for k, v in cadence.items()}, names=["view"]).reset_index(level=0)
for name, table in {"timeseries_daily": daily_out, "timeseries_weekly_bucket": weekly_bucket,
                    "timeseries_cadence": cadence_out}.items():
    table.to_parquet(CLEAN_DIR / f"{name}.parquet", index=False)
    print(f"saved {name}.parquet: {len(table):,} rows x {table.shape[1]} columns")

### What was done
- (fill in after the run: one paragraph: the series, the two clocks, what the decomposition and tests showed)
- (fill in after the run: the forecast result in one sentence, with the test RMSE and MAE vs. the best baseline)
- (fill in after the run: the cadence advice: "check every N days")
- (fill in after the run: the per-bucket momentum in one sentence)

### Answers to the clarifying questions
| Question | Answer |
|:--|:--|
| How many new scope listings per day and week; is there a weekly rhythm? | (fill in after the run) |
| Is the rhythm the employers' or the crawler's? | (fill in after the run) |
| Is there a trend across April-September, and by role bucket? | (fill in after the run) |
| Can next-day volume be forecast better than a naive baseline? | (fill in after the run) |
| How fast do listings close in each month? | (fill in after the run) |
| Do listings open to career changers arrive on a different rhythm? | Not tested here: `career_open` comes from EDA_SDA_3's text features. Add it as a cut once that table exists. |

### Limitations
- Six months of history: no seasonal claims, and the trend is a trend of this period only.
- The crawler's schedule and collection changes (job boards stop after May; descriptions go missing more from June) shape `first_seen`; the no-job-board view and the `posted` clock are the guards.
- `days_open` ends at the last sighting; crawl dropouts end it early.
- A forecast of arrivals says nothing about quality or fit; it is a timing aid.

### Handoff
- **Saved** (`data/clean/`): `timeseries_daily.parquet` (counts per day, four views), `timeseries_weekly_bucket.parquet` (per bucket and week, all sources and without job boards), `timeseries_cadence.parquet` (catch rates by gap between checks).
- **To `ML_Rank`:** the closing curve by month tells how much to trust "still open" at different ages; the cadence table is the basis of the "apply by" and "check every N days" copy in the app. Features known at first sight stay as in `STANDARDS.md`: weekday of `first_seen` follows the crawler, so it is a feature for when a listing appears to the app, not for employer behavior.
- **To the app:** the listing-momentum view per bucket next to BLS growth (never blended), and the "expect about N new listings" line from the chosen forecast.
- **Open:** repeat section 4 on `first_seen_no_board` if 2e shows the job-board stop moves the series; add the `career_open` cut once EDA_SDA_3 exists.